In [ ]:
# 1. Safe Library Installation (Uses active environment cache)
%pip install -q transformers accelerate datasets

# 2. Prevent HuggingFace from flooding your 20GB disk space
import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"

# 3. Hardware Verification Setup
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 System active. Current Device: {device.upper()}")
print(f"📊 Total available GPUs detected: {torch.cuda.device_count()}")

if torch.cuda.device_count() > 1:
    print("💡 Pro-Tip: You are running dual T4 GPUs. Use torch.nn.DataParallel() to maximize VRAM!")

Note: you may need to restart the kernel to use updated packages.
🚀 System active. Current Device: CPU
📊 Total available GPUs detected: 0


c:\Users\ADEGOKE\Desktop\DS-ML-AI\E-Commerce Risk And Demand Intelligent System\.kaggle-env\Scripts\python.exe: No module named pip


: 

In [ ]:
## Kaggle Cloud Path


from pathlib import Path
import pandas as pd

input_root = Path("/kaggle/input")
target_file = "nigeria_ecommerce_major_project_25000.csv"

if not input_root.exists():
    raise RuntimeError(
        "This cell must run on Kaggle, not the local VS Code kernel. "
        "Push the notebook with 'kaggle kernels push -p .' and run it on Kaggle."
    )

dataset_path = next(input_root.rglob(target_file), None)

if dataset_path is None:
    raise FileNotFoundError(
        f"{target_file} was not found under {input_root}. "
        "Confirm that the Kaggle dataset is attached to this Kernel."
    )

print(f"Using Kaggle dataset: {dataset_path}")
df = pd.read_csv(dataset_path)
df.head()

In [1]:
# Imports and reproducibility setup
from pathlib import Path
import json
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")



df = pd.read_csv('../data/nigeria_ecommerce_25000.csv')

print("Shape:", df.shape)
display(df.head())

Shape: (25000, 31)


,order_id,customer_id,order_timestamp,order_date,city,customer_type,customer_tenure_days,previous_orders,prior_cancellations,email_verified,...,shipping_fee,delivery_distance_km,estimated_delivery_days,payment_method,gross_amount,discount_amount,total_amount,order_status,actual_delivery_days,customer_rating
0,1,13681,2024-03-06 05:49:07,2024-03-06,Ilorin,New,6,0,0,True,...,1900.0,25.4,4,Card,8500.0,850.0,9550.0,Delivered,5.0,4.0
1,2,12075,2025-07-19 18:17:04,2025-07-19,Benin City,Returning,402,14,0,True,...,1900.0,25.4,5,Card,264800.0,26480.0,240220.0,Delivered,7.0,4.0
2,3,10402,2025-04-23 11:48:09,2025-04-23,Abuja,Returning,221,5,0,True,...,2500.0,38.3,5,Bank Transfer,54600.0,0.0,57100.0,Delivered,6.0,4.0
3,4,10764,2024-11-16 19:40:59,2024-11-16,Port Harcourt,Returning,178,13,1,False,...,2100.0,28.6,5,Cash on Delivery,238800.0,11940.0,228960.0,Delivered,6.0,3.0
4,5,11214,2024-11-12 12:49:09,2024-11-12,Port Harcourt,New,82,0,0,True,...,2400.0,34.9,5,Bank Transfer,11600.0,1160.0,12840.0,Cancelled,NaN,NaN


In [ ]:
# 1. Number of rows
print("Number of rows:", len(df))

# 2. Number of unique order IDs
print("Unique order IDs:", df["order_id"].nunique())

# 3. Are order IDs duplicated?
print("Duplicated order IDs:", df["order_id"].duplicated().sum())

# 4. Show duplicated orders if any
duplicated_orders = df[df["order_id"].duplicated(keep=False)].sort_values("order_id")

duplicated_orders.head(20)

In [ ]:
df["order_status"].value_counts(dropna=False)

In [ ]:
df["order_status"].value_counts(normalize=True, dropna=False) * 100

In [ ]:
pd.crosstab(
    df["order_status"],
    columns="count"
)

In [ ]:
pd.crosstab(
    df["order_status"],
    df["payment_method"],
    normalize="index"
).round(3)

In [ ]:
pd.crosstab(
    df["order_status"],
    df["payment_method"]
)

In [ ]:
pd.crosstab(
    df["order_status"],
    df["category"],
    normalize="index"
).round(3)

In [ ]:
df.groupby("order_status").agg(
    orders=("order_id", "count"),
    customers=("customer_id", "nunique"),
    avg_quantity=("quantity", "mean"),
    avg_unit_price=("unit_price", "mean"),
    avg_discount=("discount_percent", "mean"),
    avg_total_amount=("total_amount", "mean")
).round(2)

In [ ]:
df.groupby("order_status")["total_amount"].describe().round(2)

In [ ]:
df.groupby("order_status")["discount_percent"].describe().round(2)

In [ ]:
pd.crosstab(
    df["order_status"],
    df["city"],
    normalize="index"
).round(3)

In [ ]:
pd.crosstab(
    df["order_status"],
    df["category"],
    normalize="index"
).round(3)

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

print(df["order_date"].min())
print(df["order_date"].max())

In [ ]:
df.groupby(
    df["order_date"].dt.to_period("M")
)["order_status"].value_counts().unstack(fill_value=0)

In [ ]:
status_by_month = pd.crosstab(
    df["order_date"].dt.to_period("M"),
    df["order_status"],
    normalize="index"
) * 100

status_by_month.round(2)

In [ ]:
expected_total = (
    df["quantity"]
    * df["unit_price"]
    * (1 - df["discount_percent"] / 100)
)

df["total_difference"] = (
    df["total_amount"] - expected_total
)

df["total_difference"].describe()

In [ ]:
df[["quantity",
    "unit_price",
    "discount_percent",
    "total_amount",
    "total_difference"]].head(20)

In [ ]:
print(
    "Rows matching formula:",
    (df["total_difference"].abs() < 0.01).sum()
)

print(
    "Total rows:",
    len(df)
)

In [ ]:
print(df.columns.tolist())

In [ ]:
expected_total = (
    df["gross_amount"]
    - df["discount_amount"]
    + df["shipping_fee"]
)

difference = df["total_amount"] - expected_total

print("Rows matching formula:", (difference.abs() < 0.01).sum())
print("Total rows:", len(df))

print("\nDifference summary:")
print(difference.describe())

print("\nRows that do NOT match:")
display(
    df.loc[
        difference.abs() >= 0.01,
        [
            "quantity",
            "unit_price",
            "gross_amount",
            "discount_percent",
            "discount_amount",
            "shipping_fee",
            "total_amount"
        ]
    ].head(20)
)